# Trajectory Analysis for _Suo et. al._ iterative training

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
working_directory = "/home/icb/kemal.inecik/work/codes/sctram"
sys.path.append(working_directory)

import logging
import subprocess
import gc
import os
import time
import numpy as np
import pandas as pd
import networkx as nx
import scanpy as sc
import anndata as ad
import pickle
import itertools
import torch
import tqdm
import scvi

sc.settings.verbose = 3

In [3]:
from sctram.api._lower_level import TrajectoryEvaluationAPI
from sctram.generate.real import sc_suo_developmental_complete
from sctram.input import InputTrajectories

2025-03-18 21:50:32.613 | INFO     | sctram.api._defaults_read:load_default_metrics:23 - Loaded default metrics from /home/icb/kemal.inecik/work/codes/sctram/sctram/api/_defaults.yaml
2025-03-18 21:50:32.615 | INFO     | sctram.api._defaults_read:load_default_metrics:79 - Default metrics YAML structure validated successfully.


In [4]:
# Important to have consistent figures across platforms

%matplotlib inline
%config InlineBackend.figure_format='retina'

import pickle

from networkx.drawing.nx_agraph import graphviz_layout
from matplotlib import gridspec
import matplotlib.pyplot as plt
import seaborn as sns
import colorcet as cc
from adjustText import adjust_text  
import matplotlib.patheffects as path_effects

_rcparams_path = os.path.join(working_directory, "reproducibility/figure_rcparams/rcparams.pickle")
with open(_rcparams_path, "rb") as file:
    _rcparams = pickle.load(file)
plt.rcParams.update(_rcparams)

In [26]:
print(f"CUDA used: {torch.cuda.is_available()}")

dataset_dir = "/home/icb/kemal.inecik/lustre_workspace/temp_sctram_data"
helpers_directory = os.path.join(os.getcwd(), "helper")
logs_directory = os.path.join(os.getcwd(), "logs")

CUDA used: True


## Convert models to AnnData

In [27]:
step_per_epoch = 841922 * 0.25 * 0.8 / 512
print(step_per_epoch)

epochs = list(itertools.chain(range(20, 52), range(52, 100, 2), range(100, 200, 10), range(200, 421, 20)))
epochs = list(np.arange(0, 20, 0.10)) + epochs
epochs = [round(i, 2) for i in epochs]
steps = [int(i*step_per_epoch) for i in epochs]

print(np.array(steps))
print(len(steps))

328.87578125000005
[     0     32     65     98    131    164    197    230    263    295
    328    361    394    427    460    493    526    559    591    624
    657    690    723    756    789    822    855    887    920    953
    986   1019   1052   1085   1118   1151   1183   1216   1249   1282
   1315   1348   1381   1414   1447   1479   1512   1545   1578   1611
   1644   1677   1710   1743   1775   1808   1841   1874   1907   1940
   1973   2006   2039   2071   2104   2137   2170   2203   2236   2269
   2302   2335   2367   2400   2433   2466   2499   2532   2565   2598
   2631   2663   2696   2729   2762   2795   2828   2861   2894   2926
   2959   2992   3025   3058   3091   3124   3157   3190   3222   3255
   3288   3321   3354   3387   3420   3453   3486   3518   3551   3584
   3617   3650   3683   3716   3749   3782   3814   3847   3880   3913
   3946   3979   4012   4045   4078   4110   4143   4176   4209   4242
   4275   4308   4341   4374   4406   4439   4472   4505  

In [28]:
# Training is done with adata_training, but it is essentially the same as adata_suo
adata_suo = sc_suo_developmental_complete(dataset_dir=dataset_dir)
adata_training_file_path = os.path.join("/home/icb/kemal.inecik/lustre_workspace/tardis_data/processed", "dataset_complete_Suo.h5ad")
adata_training = ad.read_h5ad(adata_training_file_path)

assert (adata_suo.X != adata_training.X).sum() == 0
assert np.all(adata_suo.obs.index == adata_training.obs.index)

del adata_training
gc.collect();

2025-03-18 22:03:55.039 | WARNING  | sctram.generate.real._download:download_dataset:105 - File PosixPath('/home/icb/kemal.inecik/lustre_workspace/temp_sctram_data/suo_developmental_complete.h5ad') already exists. Skipping download.


In [29]:
overwrite = False
print(f" - Number of trained model for each method to be converted to AnnData: {len(steps)!r}")

for epoch in steps:
    for model_str in ["scanvi", "scvi"]:
        model_dir_path = os.path.join(dataset_dir, f"model_suo_incremental_training_{model_str}_epoch_{epoch}")
        output_dir_path = os.path.join(dataset_dir, f"adata_suo_incremental_training_{model_str}_epoch_{epoch}.h5ad")
        
        if overwrite or not os.path.exists(output_dir_path) or not os.path.isfile(output_dir_path):

            if not os.path.exists(model_dir_path) or not os.path.isdir(model_dir_path):
                print(f"Training is not prepared for model {model_str!r} and for epoch {epoch!r}.")
                continue
            
            if model_str == "scvi":
                vae = scvi.model.SCVI.load(model_dir_path, adata = adata_suo.copy())
            elif model_str == "scanvi":
                vae = scvi.model.SCANVI.load(model_dir_path, adata = adata_suo.copy())
    
            latent = ad.AnnData(X=vae.get_latent_representation(), obs=adata_suo.obs.copy())
            latent.write_h5ad(output_dir_path)
    
            del vae, latent
            gc.collect()
            print(f"AnnData prepared for model {model_str!r} and for epoch {epoch!r}.")
print(" - Completed.")     

 - Number of trained model for each method to be converted to AnnData: 278
 - Completed.


# Running the `sctram` package

```
#SBATCH -J {slurmjob_name}
#SBATCH -p cpu_p
#SBATCH --qos cpu_normal
#SBATCH -c 8
#SBATCH --mem=293G
#SBATCH --nice=0
#SBATCH -t 11:50:00
#SBATCH -o {log_file}
#SBATCH -e {log_file}

#SBATCH -J {slurmjob_name}
#SBATCH -p gpu_p
#SBATCH --qos=gpu_normal
#SBATCH --gres=gpu:1
#SBATCH -c 6
#SBATCH --mem=293G
#SBATCH --nice=0
#SBATCH -t 11:50:00
#SBATCH -o {log_file}
#SBATCH -e {log_file}
```

In [30]:
len(steps) * 5

1390

In [31]:
input_trajectories_path_litc_1_name = f"adata_suo_input_haematopoeitic_lineage_litc_1.pkl"
input_trajectories_path_litc_2_name = f"adata_suo_input_haematopoeitic_lineage_litc_2.pkl"
input_trajectories_path_name = f"adata_suo_input_haematopoeitic_lineage.pkl"

itpn_list = [input_trajectories_path_name] #, input_trajectories_path_litc_1_name, input_trajectories_path_litc_2_name]
use_reps = ["scanvi", "scvi"]
trajectory_list = ['early_b_cells', 'erythroid_megakaryocyte', 'granulocyte_monocytes', 'neutrophil', 'haematopoeitic_lineage']  # None

override = False
lineage = "Haematopoeitic_lineage"
count = 0
lineage_part = lineage.replace("_lineage", "").lower()

for use_rep in use_reps:
    
    for epoch in steps:

        adata_path = os.path.join(dataset_dir, f"adata_suo_incremental_training_{use_rep}_epoch_{epoch}.h5ad")        
        if os.path.exists(adata_path) and os.path.isfile(adata_path):
    
            for itpn in itpn_list:
                itpn_base = os.path.splitext(itpn)[0]
                itp = os.path.join(dataset_dir, itpn)
                with open(itp, "rb") as _file:
                    litc = pickle.load(_file)
                
                for trajectory in sorted(litc.graph["trajectories"]):

                    if isinstance(trajectory_list, list) and trajectory not in trajectory_list:
                        continue
                    
                    output_file = os.path.join(dataset_dir, f"_metric_adata_suo_iterative_{lineage_part}_{use_rep}_{epoch}_{itpn_base}_{trajectory}.pickle")
                    log_file = os.path.join(logs_directory, f"slurm_out_metric_adata_suo_iterative_{lineage_part}_{use_rep}_{epoch}_{itpn_base}_{trajectory}.log")
                    slurmjob_name = f"sctram_iter_{use_rep}_{epoch}"

                    if override or not os.path.exists(output_file) or not os.path.isfile(output_file):
                        try:
                            slurm_script = f"""#!/bin/bash
#SBATCH -J {slurmjob_name}
#SBATCH -p cpu_p
#SBATCH --qos cpu_normal
#SBATCH -c 8
#SBATCH --mem=293G
#SBATCH --nice=0
#SBATCH -t 11:50:00
#SBATCH -o {log_file}
#SBATCH -e {log_file}

source activate sctram_dev_env
python -u {os.path.join(helpers_directory, 'suo_sctram_iterative.py')} --lineage "{lineage}" --use_rep "{use_rep}" --epoch "{epoch}" --itpn_base "{itpn_base}" --trajectory "{trajectory}"
            """
                            script_name = os.path.join(logs_directory, f"slurm_job_metric_adata_suo_iterative_{lineage_part}_{use_rep}_{epoch}_{itpn_base}_{trajectory}.sh")
                            with open(script_name, "w") as f:
                                f.write(slurm_script)
            
                            print(f"Submitted job {count+1!r} of {lineage!r} with {use_rep!r} of trajectory {trajectory!r} for epoch {epoch!r} for {itpn_base!r} of {trajectory!r}")
                            subprocess.run(["sbatch", script_name])
                            count += 1
                            # if count > 1:
                            #     raise ValueError
                        finally:
                            # time.sleep(0.1)
                            os.remove(script_name)

                    else:
                        print(f"Already exist: {lineage!r} with {use_rep!r} of trajectory {trajectory!r} for epoch {epoch!r} for {itpn_base!r} of {trajectory!r}")
            
        else:
            print(f"AnnData is not prepared for model {use_rep!r} and for epoch {epoch!r}.")

print(f" - Number of jobs submitted: {count}")

# Running the `scIB` package

In [32]:
override = False
count = 0

trajectory_list += ["all_lineage_no_filter"]

for use_rep in use_reps:
    
    for epoch in steps:

        adata_path = os.path.join(dataset_dir, f"adata_suo_incremental_training_{use_rep}_epoch_{epoch}.h5ad")        
        if os.path.exists(adata_path) and os.path.isfile(adata_path):
    
            for itpn in itpn_list:
                itpn_base = os.path.splitext(itpn)[0]
                itp = os.path.join(dataset_dir, itpn)
                with open(itp, "rb") as _file:
                    litc = pickle.load(_file)

                the_trajectories = sorted(litc.graph["trajectories"])
                if "all_lineage_no_filter" in trajectory_list:
                    the_trajectories.append("all_lineage_no_filter")
                
                for trajectory in the_trajectories:

                    if isinstance(trajectory_list, list) and trajectory not in trajectory_list:
                        continue
                    
                    output_file = os.path.join(dataset_dir, f"_metric_adata_suo_iterative_{lineage_part}_{use_rep}_{epoch}_{itpn_base}_{trajectory}_scib.pickle")
                    log_file = os.path.join(logs_directory, f"slurm_out_metric_adata_suo_iterative_{lineage_part}_{use_rep}_{epoch}_{itpn_base}_{trajectory}_scib.log")
                    slurmjob_name = f"scib_iter_{use_rep}_{epoch}"

                    if override or not os.path.exists(output_file) or not os.path.isfile(output_file):
                        try:
                            slurm_script = f"""#!/bin/bash
#SBATCH -J {slurmjob_name}
#SBATCH -p cpu_p
#SBATCH --qos cpu_normal
#SBATCH -c 32
#SBATCH --mem=80G
#SBATCH --nice=0
#SBATCH -t 11:50:00
#SBATCH -o {log_file}
#SBATCH -e {log_file}

source activate sctram_dev_env
python -u {os.path.join(helpers_directory, 'suo_scib_iterative.py')} --lineage "{lineage}" --use_rep "{use_rep}" --epoch "{epoch}" --itpn_base "{itpn_base}" --trajectory "{trajectory}"
            """
                            script_name = os.path.join(logs_directory, f"slurm_job_metric_adata_suo_iterative_{lineage_part}_{use_rep}_{epoch}_{itpn_base}_{trajectory}_scib.sh")
                            with open(script_name, "w") as f:
                                f.write(slurm_script)
            
                            print(f"Submitted job for scIB {count+1!r} of {lineage!r} with {use_rep!r} of trajectory {trajectory!r} for epoch {epoch!r} for {itpn_base!r} of {trajectory!r}")
                            subprocess.run(["sbatch", script_name])
                            count += 1
                            # if count > 1:
                            #     raise ValueError
                        finally:
                            # time.sleep(0.1)
                            os.remove(script_name)

                    else:
                        pass
                        # print(f"scIB calculations already exist: {lineage!r} with {use_rep!r} of trajectory {trajectory!r} for epoch {epoch!r} for {itpn_base!r} of {trajectory!r}")
            
        else:
            raise ValueError(f"AnnData is not prepared for model {use_rep!r} and for epoch {epoch!r}.")

print(f" - Number of jobs submitted: {count}")

Submitted job for scIB 1 of 'Haematopoeitic_lineage' with 'scanvi' of trajectory 'all_lineage_no_filter' for epoch 0 for 'adata_suo_input_haematopoeitic_lineage' of 'all_lineage_no_filter'
Submitted batch job 34555880
Submitted job for scIB 2 of 'Haematopoeitic_lineage' with 'scanvi' of trajectory 'all_lineage_no_filter' for epoch 32 for 'adata_suo_input_haematopoeitic_lineage' of 'all_lineage_no_filter'
Submitted batch job 34555881
Submitted job for scIB 3 of 'Haematopoeitic_lineage' with 'scanvi' of trajectory 'all_lineage_no_filter' for epoch 65 for 'adata_suo_input_haematopoeitic_lineage' of 'all_lineage_no_filter'
Submitted batch job 34555882
Submitted job for scIB 4 of 'Haematopoeitic_lineage' with 'scanvi' of trajectory 'all_lineage_no_filter' for epoch 98 for 'adata_suo_input_haematopoeitic_lineage' of 'all_lineage_no_filter'
Submitted batch job 34555883
Submitted job for scIB 5 of 'Haematopoeitic_lineage' with 'scanvi' of trajectory 'all_lineage_no_filter' for epoch 131 for 'a

In [34]:
1

1